In [1]:
import pandas as pd

# Load the wide spread table
df_wide = pd.read_csv("Agg_Dirty_Spread.csv")

# Normalize the Date column (handles BOM if present) and parse as datetime
df_wide.columns = df_wide.columns.str.strip().str.replace("\ufeff", "", regex=False)
df_wide["Date"] = pd.to_datetime(df_wide["Date"])

# Wide -> long: one row per (Date, CUSIP)
df_long = (
    df_wide
    .melt(id_vars="Date", var_name="CUSIP", value_name="Spread")
    .dropna(subset=["Spread"])
)

# Enforce types
df_long["CUSIP"] = df_long["CUSIP"].astype(str).str.strip()
df_long["Spread"] = pd.to_numeric(df_long["Spread"], errors="coerce")
df_long = df_long.dropna(subset=["Spread"])

# Sanity checks before exporting
print(df_long.shape)
print(df_long.dtypes)
print(df_long.head())
print("unique CUSIPs:", df_long["CUSIP"].nunique())
print("date range:", df_long["Date"].min(), "→", df_long["Date"].max())

# Write the file you'll upload to BigQuery
df_long.to_csv("Agg_Spread_Long.csv", index=False)

(148429, 3)
Date      datetime64[ns]
CUSIP             object
Spread           float64
dtype: object
        Date      CUSIP      Spread
0 2024-03-01  0010EPAF5  134.502334
1 2024-03-04  0010EPAF5  138.433861
2 2024-03-05  0010EPAF5  140.019836
3 2024-03-06  0010EPAF5  140.262941
4 2024-03-07  0010EPAF5  134.621923
unique CUSIPs: 303
date range: 2024-03-01 00:00:00 → 2026-02-26 00:00:00
